In [ ]:
import emcee
import numpy as np

# -------------------------------------------------------------------------
# 1. Synthetic Data Generation (Toy Linear Model: y = m*x + c)
# -------------------------------------------------------------------------
np.random.seed(42)
N_data = 50
x_true = np.sort(np.random.uniform(0.0, 10.0, N_data))
sigma_true = np.random.uniform(0.2, 0.6, N_data)

m_true = 1.5
c_true = 2.0
y_obs = m_true * x_true + c_true + np.random.normal(0.0, sigma_true)


# -------------------------------------------------------------------------
# 2. Likelihood Functions
# -------------------------------------------------------------------------
def pointwise_log_likelihood(theta, x, y, yerr):
    """Return a 1D array of shape (N,) containing ln p(y_i | theta).

    Essential for WAIC: keeps each residual unsummed.
    """
    m, c = theta
    model = m * x + c
    # Gaussian pointwise log-likelihood:
    # ln p(y_i | theta) = -0.5 * [ ((y_i - mu_i)/sigma_i)^2 + ln(2*pi*sigma_i^2) ]
    chi2_i = ((y - model) / yerr) ** 2
    norm_const = np.log(2.0 * np.pi * (yerr**2))
    return -0.5 * (chi2_i + norm_const)


def log_prior(theta):
    m, c = theta
    if -5.0 < m < 5.0 and -10.0 < c < 10.0:
        return 0.0
    return -np.inf


def log_posterior(theta, x, y, yerr):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    # emcee needs a single scalar log-likelihood
    ll_vec = pointwise_log_likelihood(theta, x, y, yerr)
    return lp + np.sum(ll_vec)


# -------------------------------------------------------------------------
# 3. Sampling Setup
# -------------------------------------------------------------------------
ndim = 2
nwalkers = 32
nsteps = 100000
burn_in = 10000

# Initial walker positions around an initial guess
initial_guess = np.array([1.0, 1.0])
pos0 = initial_guess + 1e-3 * np.random.randn(nwalkers, ndim)

# Run MCMC
sampler = emcee.EnsembleSampler(
    nwalkers, ndim, log_posterior, args=(x_true, y_obs, sigma_true)
)
print("Running MCMC...")
sampler.run_mcmc(pos0, nsteps, progress=True)

# -------------------------------------------------------------------------
# 4. Extracting Posterior Draws (theta^s)
# -------------------------------------------------------------------------
# Flatten across walkers and discard burn-in
flat_samples = sampler.get_chain(discard=burn_in, flat=True)
total_available_samples = flat_samples.shape[0]

# Thin the chain to S draws (e.g. 1000) to keep WAIC post-processing fast
S = 1000
subsample_indices = np.random.choice(
    total_available_samples, size=S, replace=False
)
posterior_draws = flat_samples[subsample_indices]  # Shape: (S, ndim)

# -------------------------------------------------------------------------
# 5. Build the Pointwise Log-Likelihood Matrix for WAIC
# -------------------------------------------------------------------------
# Matrix shape: (S, N_data) where element [s, i] is ln p(y_i | theta^s)
log_lik_matrix = np.empty((S, N_data))

for s_idx, theta_s in enumerate(posterior_draws):
    log_lik_matrix[s_idx, :] = pointwise_log_likelihood(
        theta_s, x_true, y_obs, sigma_true
    )


print("\nReady for WAIC calculation:")
print(f"Posterior samples (S): {log_lik_matrix.shape[0]}")
print(f"Data points        (N): {log_lik_matrix.shape[1]}")
print(f"Matrix shape (S, N)   : {log_lik_matrix.shape}")

# -------------------------------------------------------------------------
# TODO: Plug in your WAIC calculation below using `log_lik_matrix`
# -------------------------------------------------------------------------

def WAIC(lppd, p_w):
    return -2.0*lppd + 2.0*p_w

def lppd(data_matrix):
    """ 
    N - number of data points
    S - number of parameters in theta
    lh - likelihood function
    y - data set (observations)
    theta - parameters
    """
    S = data_matrix.shape[0]
    lppd = []
    for n in range(data_matrix.shape[0]):
        for s in range(data_matrix.shape[1]):
            lppd[n] = ((1/S) * data_matrix[n,s])
    sum_lppd = np.sum(lppd)

    return sum_lppd

def p_w(data_matrix):
    pw = []
    """ 
    lh - likelihood function
    y - data set
    theta - parameters
    """
    for n in range(data_matrix.shape[0]):
        for s in range(data_matrix.shape[1]):
            pw[n] = np.var(data_matrix[n,s])
    sum_pw_log = np.sum(pw_log)

    return sum_pw_log

"""
y - log_like_matrix[0]
theta - log_like_matrix[1]
Nevermind working with the matrix is funky, gonna have to find a way to work with it entirely
"""







Running MCMC...


100%|██████████| 100000/100000 [00:42<00:00, 2335.73it/s]


Ready for WAIC calculation:
Posterior samples (S): 1000
Data points        (N): 50
Matrix shape (S, N)   : (1000, 50)
WAIC: 0.0007126538877128089


In [15]:
#Gemini answer:
from scipy.special import logsumexp

def compute_lppd(log_lik_matrix):
    S = log_lik_matrix.shape[0]
    #logsumexp used to easily calculate ln(sum(exp))
    lppd_i = logsumexp(log_lik_matrix, axis = 0) - np.log(S)
    return np.sum(lppd_i)

def compute_p_waic(log_lik_matrix):
    p_w_i = np.var(log_lik_matrix, ddof = 1, axis = 0)
    return np.sum(p_w_i)

def compute_waic(log_lik_matrix):
    lppd_val = compute_lppd(log_lik_matrix)
    p_w_val = compute_p_waic(log_lik_matrix)
    waic = -2.0*lppd_val + 2.0*p_w_val
    return waic, lppd_val, p_w_val

waic, lppd_val, p_w_val = compute_waic(log_lik_matrix)

print(f"lppd:     {lppd_val:.4f}")
print(f"p_w:   {p_w_val:.4f}  (effective parameters, should be ~2)")
print(f"WAIC:     {waic:.4f}")


lppd:     -18.3452
p_w:   1.8626  (effective parameters, should be ~2)
WAIC:     40.4156


In [24]:
#Practice 2:
import emcee
import numpy as np

# -------------------------------------------------------------------------
# 1. Synthetic Data: y = A * exp(-k * x) + b
# -------------------------------------------------------------------------
np.random.seed(101)
N_data = 60
x_data = np.sort(np.random.uniform(0.0, 5.0, N_data))
sigma_data = np.random.uniform(0.15, 0.35, N_data)

A_true = 4.0
k_true = 0.8
b_true = 1.2
y_data = (
    A_true * np.exp(-k_true * x_data)
    + b_true
    + np.random.normal(0.0, sigma_data)
)


# -------------------------------------------------------------------------
# 2. Likelihood & Priors
# -------------------------------------------------------------------------
def pointwise_log_likelihood(theta, x, y, yerr):
    A, k, b = theta
    model = A * np.exp(-k * x) + b
    chi2_i = ((y - model) / yerr) ** 2
    norm = np.log(2.0 * np.pi * (yerr**2))
    return -0.5 * (chi2_i + norm)  # 1D array of shape (N_data,)


def log_prior(theta):
    A, k, b = theta
    if (0.0 < A < 10.0) and (0.01 < k < 3.0) and (-2.0 < b < 5.0):
        return 0.0
    return -np.inf


def log_posterior(theta, x, y, yerr):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + np.sum(pointwise_log_likelihood(theta, x, y, yerr))


# -------------------------------------------------------------------------
# 3. MCMC Sampling
# -------------------------------------------------------------------------
ndim = 3
nwalkers = 32
nsteps = 2000
burn_in = 600

init_pos = np.array([3.5, 1.0, 1.0]) + 1e-2 * np.random.randn(nwalkers, ndim)

sampler = emcee.EnsembleSampler(
    nwalkers, ndim, log_posterior, args=(x_data, y_data, sigma_data)
)
print("Running practice MCMC...")
sampler.run_mcmc(init_pos, nsteps, progress=True)

# -------------------------------------------------------------------------
# 4. Extract Draws & Build Pointwise Log-Likelihood Matrix
# -------------------------------------------------------------------------
flat_samples = sampler.get_chain(discard=burn_in, flat=True)
S = 1000
idx = np.random.choice(flat_samples.shape[0], size=S, replace=False)
draws = flat_samples[idx]  # Shape: (S, ndim)

# Shape: (S, N_data) -> (1000, 60)
log_lik_matrix = np.empty((S, N_data))
for s, theta_s in enumerate(draws):
    log_lik_matrix[s, :] = pointwise_log_likelihood(
        theta_s, x_data, y_data, sigma_data
    )

print("\nReady for practice calculation:")
print(f"Matrix shape: {log_lik_matrix.shape}")

# -------------------------------------------------------------------------
# Practice Task:
# Implement your WAIC, lppd, and p_waic functions below using `log_lik_matrix`
# -------------------------------------------------------------------------
def compute_lppdv2(log_lik_matrix):
    """ 
    Couple notes on this:
    lppd - log pointwise posterior predictive density. Measure of how well fitted statistical model predicts observed data.
    """
    S = log_lik_matrix.shape[0] #1000
    c = np.max(log_lik_matrix, axis = 0) #finds max loglike for each samples parameter for all samples
    #subtracting c prevents underflow to 0 which could crash the program.
    lppd_i = c + np.log(np.sum(np.exp(log_lik_matrix - c), axis = 0)) - np.log(S)
    return np.sum(lppd_i)

def compute_p_waicv2(log_lik_matrix):
    S = log_lik_matrix.shape[0] #1000
    N = log_lik_matrix.shape[1] #60
    pwi = np.var(log_lik_matrix, ddof = 1, axis = 0)
    return np.sum(pwi)

def compute_waicv2(log_lik_matrix):
    lppd = compute_lppdv2(log_lik_matrix)
    pw = compute_p_waicv2(log_lik_matrix)
    waic = -2.0 * lppd + 2.0 * pw
    return waic

print(f"WAIC: {compute_waicv2(log_lik_matrix)}")
print(f"LPPD: {compute_lppdv2(log_lik_matrix)}")
print(f"p_waic: {compute_p_waicv2(log_lik_matrix)}")


Running practice MCMC...


100%|██████████| 2000/2000 [00:00<00:00, 2043.28it/s]


Ready for practice calculation:
Matrix shape: (1000, 60)
WAIC: 5.049890441559416
LPPD: 0.41986167588156587
p_waic: 2.944806896661274


In [19]:
#Setting up an np.matrix
import numpy as np
matrix = np.array([[1,2,3],[4,5,6]])
print(matrix)
print(matrix.shape) #first number denotes number of rows, second number denotes number of columns
print(matrix[1,2])


#Populating one using a for-loop
N = 3
S = 3
matrix = np.zeros((N,S))
for n in range(N):
    for s in range(S):
        matrix[n,s] = 1.0

print(matrix)

[[1 2 3]
 [4 5 6]]
(2, 3)
6
[[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]


In [ ]:
#The WAIC pointwise-likelihood function is given by p(y|theta) where y is our observation and theta the parameters for said observation


TypeError: object of type 'int' has no len()

In [38]:
samples = [7,8,9,20]
n_valid = len(samples)
S = min(1000, n_valid)
idx = np.random.choice(
    n_valid, size = S, replace = False
)
print(idx)

[1 3 2 0]


In [2]:
import h5py, numpy as np

p = 'MCMC_Chains/Test/wowaCDM_v2/DESI_DR2/DESI_DR2.h5'
with h5py.File(p, 'r') as f:
    lp = np.asarray(f['mcmc/log_prob'])
    
    # Count -inf or very bad likelihoods
    n_bad = np.isinf(lp).sum() + (lp < -1e6).sum()
    n_total = lp.size
    
    print(f"Bad samples: {n_bad} / {n_total} ({100*n_bad/n_total:.1f}%)")
    print(f"Log-prob range: {np.nanmin(lp):.1f} to {np.nanmax(lp):.1f}")

Bad samples: 0 / 160000 (0.0%)
Log-prob range: -9157.3 to -5.0


In [3]:
import h5py, numpy as np

p = 'MCMC_Chains/Test/wowaCDM_v2/DESI_DR2/DESI_DR2.h5'
with h5py.File(p, 'r') as f:
    chain = np.asarray(f['mcmc/chain'])
    
    # Check how many samples violate w0+wa < 0
    w0_idx = 2  # Adjust if different
    wa_idx = 3
    
    w0 = chain[:, :, w0_idx].flatten()
    wa = chain[:, :, wa_idx].flatten()
    
    violating = (w0 + wa) >= -0.001  # Allow tiny numerical error
    print(f"Samples with w0+wa ≥ 0: {violating.sum()} / {len(wa)} ({100*violating.mean():.1f}%)")
    
    if violating.mean() > 0.05:
        print("⚠️ VIOLATION RATE >5%: Coupled restriction not enforced!")

Samples with w0+wa ≥ 0: 160000 / 160000 (100.0%)
⚠️ VIOLATION RATE >5%: Coupled restriction not enforced!


In [4]:
import h5py, numpy as np
from Kosmulator_main.MCMC_setup import main  # or load CONFIG directly
from Kosmulator_main import Config

# Get CONFIG (same way MCMC does)
# ... or just print from inside the chain:

p = 'MCMC_Chains/Test/wowaCDM_v2/DESI_DR2/DESI_DR2.h5'
with h5py.File(p, 'r') as f:
    # Print metadata if stored
    if 'parameter_names' in f.attrs:
        names = f.attrs['parameter_names']
        print(f"Parameter names: {names}")
    else:
        print("No parameter names stored in HDF5.")

# Fallback: manually reconstruct from CONFIG
from Kosmulator_main.Config import create_config
import User_defined_modules as UDM

models = UDM.Get_model_names(["wowaCDM_v2"])
CONFIG, _ = create_config(
    models=models,
    observation=[["DESI_DR2"]],
    model_name=["wowaCDM_v2"],
    nwalkers=32,
    nsteps=100,
    burn=10,
    prior_limits={
        "Omega_m": (0.1, 0.5),
        "w0": (-1.5, -0.1),
        "wa": (-1.5, 0.5),
        "H_0": (60, 75),
        "r_d": (140, 150),
    },
    reference_values={},
)

params = CONFIG["wowaCDM_v2"]["parameters"][0]
print(f"Parameter order for obs_index=0: {params}")

# Now check with correct indices
for i, name in enumerate(params):
    print(f"  Index {i}: {name}")

WARNING | Added ['H_0', 'r_d'] to parameters for ['DESI_DR2'] in model wowaCDM_v2


No parameter names stored in HDF5.
Parameter order for obs_index=0: ['Omega_m', 'w0', 'wa', 'H_0']
  Index 0: Omega_m
  Index 1: w0
  Index 2: wa
  Index 3: H_0


In [16]:
#Exercise 3a 
from scipy.integrate import quad
import numpy as np

#integrating the function X(z) = \int(cdz/ H(z)) for z = 0.5 and z = 1100

def function(z):
    Om = 0.315
    Ol = 0.685
    Ho = 67.4
    c = 3e5
    H = np.sqrt(Ho**2 * (Om*(1+z)**3 + Ol))
    return np.pow(H, -1)*c

#integrand limits:
a = 0
z1 = 0.5
z2 = 1100

X1, error1 = quad(function, a, z1)
X2, error2 = quad(function, a, z2)
print(f"X1: {X1}")
print(f"X2: {X2}")

#Computing dTheta = rd/xZ

X1: 1952.7379032581355
X2: 13944.162391146356


BELOW IS MARCEL'S CHAINS with statistical analysis performed onto them

In [ ]:
"""
This code is from Claude to help analyse Marcel's chains
"""
#!/usr/bin/env python3
"""
Simple script to read Kosmulator MCMC chains from HDF5 files.
"""

import h5py
import numpy as np
from pathlib import Path

def inspect_chain(chain_file):
    """Show what's inside a chain file."""
    with h5py.File(chain_file, 'r') as f:
        print(f"\n{'='*60}")
        print(f"Chain: {chain_file.name}")
        print(f"{'='*60}")
        print(f"Contents: {list(f.keys())}\n")
        
        # Explore each top-level item
        for key in f.keys():
            item = f[key]
            if isinstance(item, h5py.Dataset):
                print(f"{key}:")
                print(f"  Shape: {item.shape}")
                print(f"  Dtype: {item.dtype}")
                print()

def load_chain(chain_file):
    """
    Load chain data from HDF5 file.
    Returns: samples, log_likelihood, parameter_names
    """
    with h5py.File(chain_file, 'r') as f:
        # The exact keys depend on how your chains were saved
        # Common patterns:
        if 'chain' in f:
            samples = f['chain'][:]  # Shape: (n_iterations, n_walkers, n_params) or (n_samples, n_params)
        elif 'samples' in f:
            samples = f['samples'][:]
        else:
            # List what's available
            print(f"Available datasets: {list(f.keys())}")
            raise KeyError("Could not find 'chain' or 'samples' dataset")
        
        # Log-likelihood (may be in different places)
        log_like = None
        if 'log_like' in f:
            log_like = f['log_like'][:]
        elif 'loglike' in f:
            log_like = f['loglike'][:]
        elif 'likelihood' in f:
            log_like = f['likelihood'][:]
        
        # Parameter names
        param_names = None
        if 'names' in f:
            param_names = [name.decode() if isinstance(name, bytes) else name 
                          for name in f['names'][:]]
        
        return samples, log_like, param_names


# ============================================================================
# TEST: Inspect your chains
# ============================================================================

if __name__ == "__main__":
    # Point to one of your chains
    chains_dir = Path("MCMC_Chains/Marcel_Chains/Free/NonLinear_IDE_2/DESI_DR2_PantheonP_SH0ES")
    chain_file = chains_dir / "DESI_DR2+PantheonP_SH0ES.h5"

    #chains_dir_1_lcdm = Path("MCMC_Chains/Marcel_Chains/Free/LCDM_v/DESI_DR2_PantheonP_SH0ES")
    #chain_file_1_lcdm = chains_dir_1_lcdm / "DESI_DR2+PantheonP_SH0ES.h5"


    
    if chain_file.exists():
        # First: see what's inside
        inspect_chain(chain_file)
        
        # Then: try to load it
        try:
            samples, log_like, names = load_chain(chain_file)
            print(f"\nLoaded successfully!")
            print(f"Samples shape: {samples.shape}")
            print(f"Log-likelihood shape: {log_like.shape if log_like is not None else 'N/A'}")
            print(f"Parameters: {names if names else 'N/A'}")
        except Exception as e:
            print(f"Error loading: {e}")
    else:
        print(f"Chain file not found: {chain_file}")


Chain: DESI_DR2+PantheonP_SH0ES.h5
Contents: ['log_like', 'mcmc']

log_like:
  Shape: (12000000,)
  Dtype: float64

Available datasets: ['log_like', 'mcmc']
Error loading: "Could not find 'chain' or 'samples' dataset"


Inspecting chain file...

INSPECTING: DESI_DR2+PantheonP_SH0ES.h5

📊 log_like: shape=(12000000), dtype=float64
📁 mcmc/ (Group)
  📊 accepted: shape=(120), dtype=float64
  📊 blobs: shape=(100000×120), dtype=float64
  📊 chain: shape=(100000×120×6), dtype=float64
  📊 log_prob: shape=(100000×120), dtype=float64

EXTRACTING DATA

✓ Found chain data:
  Iterations: 100000
  Walkers: 120
  Parameters: 6

✓ Applied burn-in=0, thin=1
  Remaining iterations: 100000
✓ Found log-likelihood: shape=(100000, 120)

✓ Flattened:
  Total samples: 12000000
  Log-like matches samples: True
✓ Parameter names: ['param_0', 'param_1', 'param_2']

DATA SUMMARY
Samples shape: (12000000, 6)
Log-likelihood shape: (12000000,)
Parameters: 6
Sample range: [-19.849354, 178.678426]
Log-like range: [-19080.032984, 0.000000]
Log-like mean: -200.378444

MODEL STATISTICS

DIC Analysis:
  DIC = 430694.5782
  D_bar = 400.7569
  p_D = 430293.8213

WAIC Analysis:
  WAIC = 860988.3994
  D_bar = 400.7569
  p_WAIC = 430293.8213

P